In [0]:
SRC_TABLE = "workspace.insurance_fraud.auto_insurance_fraud"   # READ-ONLY warehouse table
DM = "workspace.dm_final_project"                               # ALL derived work goes here
SEED = 42
assert "insurance_fraud" not in DM, "Never write to the warehouse schema"

from pyspark.sql import functions as F, Window
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DM}")
except Exception as e:
    print("Schema create skipped (probably already exists / no permission):", e)

## 1. Extract

In [0]:
raw = spark.table(SRC_TABLE)
raw_count = raw.count()
print(f"Raw rows: {raw_count:,}")
raw.printSchema()
display(raw.limit(10))

## 2. Profile raw data quality

In [0]:
null_df = raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).toPandas().T
null_df.columns = ["null_count"]
null_df["null_pct"] = (100 * null_df["null_count"] / raw_count).round(2)
display(null_df.reset_index().rename(columns={"index": "column"}))

dup_ids = raw.groupBy("claim_id").count().filter("count > 1").count()
exact_dups = raw_count - raw.dropDuplicates().count()
print(f"claim_ids appearing more than once: {dup_ids:,} | exact duplicate rows: {exact_dups:,}")

## 3. Transform
Rules (each is logged in `dq_report`):
1. Cast every column to the type in the data dictionary.
2. Standardize strings: trim, upper-case state/provider/attorney, collapse whitespace, blank -> NULL.
3. De-duplicate on `claim_id` (keep the most-amended version = latest claim state).
4. Range checks: out-of-range values are set to NULL (not dropped) so the claim stays reviewable; imputation decisions are made later and documented in the preparation notebook.
5. Rows with a missing/invalid target label are dropped (cannot be used for supervised learning).

In [0]:
EXPECTED_TYPES = {
    "claim_id": "string", "claim_date": "date", "state": "string", "insured_age": "int",
    "policy_inception_days": "int", "num_people_in_car": "int", "num_different_surnames": "int",
    "report_delay_hours": "int", "medical_provider": "string", "attorney_firm": "string",
    "incident_latitude": "double", "incident_longitude": "double", "incident_time_hour": "int",
    "prior_claims_6mo": "int", "vehicle_age_years": "int", "damage_amount": "double",
    "claim_amendments": "int", "initial_damage_amount": "double", "medical_visits_count": "int",
    "days_to_attorney": "int", "settlement_inquiries": "int",
    "is_organized_fraud": "int", "is_opportunistic_fraud": "int", "claim_notes": "string",
}
missing_cols = set(EXPECTED_TYPES) - set(raw.columns)
assert not missing_cols, f"Columns missing from source: {missing_cols}"

df = raw.select([F.col(c).cast(t).alias(c) for c, t in EXPECTED_TYPES.items()])

# string standardization
def blank_to_null(c):
    return F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))

for c in ["state", "medical_provider", "attorney_firm"]:
    df = df.withColumn(c, F.upper(F.regexp_replace(blank_to_null(c), r"\s+", " ")))
df = df.withColumn("claim_notes", blank_to_null("claim_notes"))
df = df.withColumn("claim_id", F.upper(F.trim("claim_id")))

dq_log = []

In [0]:
# --- De-duplicate on claim_id ---------------------------------------------------
n_before = df.count()
w = Window.partitionBy("claim_id").orderBy(F.col("claim_amendments").desc_nulls_last(), F.col("claim_date").desc_nulls_last())
df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
n_after = df.count()
dq_log.append(dict(rule="duplicate claim_id", column="claim_id", violations=n_before - n_after, action="kept most-amended row"))

# --- Invalid / missing labels ----------------------------------------------------
bad_label = (~F.col("is_organized_fraud").isin(0, 1)) | (~F.col("is_opportunistic_fraud").isin(0, 1)) \
            | F.col("is_organized_fraud").isNull() | F.col("is_opportunistic_fraud").isNull()
n_bad = df.filter(bad_label).count()
df = df.filter(~bad_label)
dq_log.append(dict(rule="target not in {0,1} or NULL", column="is_*_fraud", violations=n_bad, action="row dropped"))

# --- Null missing claim_id ---------------------------------------------------------
n_noid = df.filter(F.col("claim_id").isNull()).count()
df = df.filter(F.col("claim_id").isNotNull())
dq_log.append(dict(rule="claim_id NULL", column="claim_id", violations=n_noid, action="row dropped"))

In [0]:
#--- Range / validity checks -> set to NULL ---------------------------------------
# condition = the VALID range. Adjust bounds with the business if needed.
RANGE_RULES = {
    "insured_age":            (F.col("insured_age").between(16, 100),            "age 16-100"),
    "policy_inception_days":  (F.col("policy_inception_days") >= 0,               ">= 0"),
    "num_people_in_car":      (F.col("num_people_in_car").between(1, 12),         "1-12"),
    "num_different_surnames": (F.col("num_different_surnames") >= 0,              ">= 0"),
    "report_delay_hours":     (F.col("report_delay_hours") >= 0,                  ">= 0"),
    "incident_latitude":      (F.col("incident_latitude").between(24, 48),        "continental US NE/SE lat"),
    "incident_longitude":     (F.col("incident_longitude").between(-92, -66),     "continental US NE/SE lon"),
    "incident_time_hour":     (F.col("incident_time_hour").between(0, 23),        "0-23"),
    "prior_claims_6mo":       (F.col("prior_claims_6mo") >= 0,                    ">= 0"),
    "vehicle_age_years":      (F.col("vehicle_age_years").between(0, 40),         "0-40"),
    "damage_amount":          (F.col("damage_amount") >= 0,                       ">= 0"),
    "initial_damage_amount":  (F.col("initial_damage_amount") >= 0,               ">= 0"),
    "claim_amendments":       (F.col("claim_amendments") >= 0,                    ">= 0"),
    "medical_visits_count":   (F.col("medical_visits_count") >= 0,                ">= 0"),
    "days_to_attorney":       (F.col("days_to_attorney") >= 0,                    ">= 0"),
    "settlement_inquiries":   (F.col("settlement_inquiries") >= 0,                ">= 0"),
}
for col, (valid, desc) in RANGE_RULES.items():
    n_viol = df.filter(F.col(col).isNotNull() & ~valid).count()
    df = df.withColumn(col, F.when(F.col(col).isNull() | valid, F.col(col)).otherwise(F.lit(None).cast(EXPECTED_TYPES[col])))
    dq_log.append(dict(rule=f"{col} outside valid range ({desc})", column=col, violations=n_viol, action="set to NULL"))

# logical consistency checks (flag only; do not modify)
n_surn = df.filter(F.col("num_different_surnames") > F.col("num_people_in_car")).count()
dq_log.append(dict(rule="surnames > occupants", column="num_different_surnames", violations=n_surn, action="flag only (investigate)"))
n_future = df.filter(F.col("claim_date") > F.current_date()).count()
dq_log.append(dict(rule="claim_date in the future", column="claim_date", violations=n_future, action="flag only (investigate)"))

## 4. Load into the data mart

In [0]:
clean_count = df.count()
reconciled = raw_count - sum(r["violations"] for r in dq_log if r["action"] in ("kept most-amended row", "row dropped"))
print(f"raw={raw_count:,} clean={clean_count:,} expected={reconciled:,}")
assert clean_count == reconciled, "Row reconciliation failed - investigate before loading"

(df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.claims_clean"))
spark.sql(f"COMMENT ON TABLE {DM}.claims_clean IS 'Cleaned IronKey auto claims. Source: {SRC_TABLE}. Built by 01_etl.'")

dq = spark.createDataFrame(dq_log).withColumn("pct_of_raw", F.round(100 * F.col("violations") / F.lit(raw_count), 3)) \
          .withColumn("run_ts", F.current_timestamp())
dq.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.dq_report")
display(dq.orderBy(F.col("violations").desc()))

In [0]:
# Post-load validation
chk = spark.table(f"{DM}.claims_clean")
print("rows:", chk.count(), "| distinct claim_id:", chk.select("claim_id").distinct().count())
display(chk.groupBy("is_organized_fraud", "is_opportunistic_fraud").count())